Quality control of Cordillera Vilcanota AWS datasets
Reproducible processing for Ausangate, Murmurani Alto, Osjollo Anante, Quelccaya, and Lake Sibinacocha.

The workflow standardizes timestamps and variable names, applies documented quality-control filters, calculates relative humidity with respect to ice (RHi), applies precipitation undercatch corrections, and writes one final QC file per station. Missing observations are not interpolated.

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

In [10]:


DATA_DIR = Path(r"D:\UNR\South America\Data")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

def hourly_index(df, time="TIMESTAMP", fmt=None):
    """Parse timestamps, remove duplicates, sort, and insert missing hourly timestamps."""
    df = df.copy()
    df[time] = pd.to_datetime(df[time], format=fmt, errors="coerce")
    df = df.dropna(subset=[time]).sort_values(time).drop_duplicates(time, keep="last").set_index(time)
    return df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="h")).rename_axis("TIMESTAMP")

def rh_water_to_ice(rh, t):
    """Convert RH with respect to water to RH with respect to ice (%)."""
    ew = 6.112 * np.exp(17.62 * t / (t + 243.04))
    ei = 6.112 * np.exp(22.46 * t / (t + 272.62))
    return np.minimum(rh * ew / ei, 100)

def correct_precipitation(df, shield, temp="Tair", precip="Precip", wind="WS"):
    """Apply phase-dependent gauge-undercatch correction used in the original workflow."""
    out = df.copy()
    T, P, U = out[temp], out[precip], out[wind]
    phase = np.select([T > 2, T >= 0], ["liquid", "mixed"], default="solid").astype(object)
    phase[T.isna()] = "none"
    out["modelled_phase"] = phase
    corr = P.copy()

    # Parameters: catch efficiency = a*exp(-b*U)+c; wind is capped at Umax.
    pars = {
        "none":   {"mixed": (0.641, 0.236, 0.356, 7.2), "solid": (0.860, 0.371, 0.229, 7.2)},
        "single": {"mixed": (0.668, 0.132, 0.339, 7.2), "solid": (0.728, 0.230, 0.336, 7.2)},
        "double": {"mixed": (0.260, 1.512, 0.950, 6.1), "solid": (1.068, 0.049, 0.000, 6.1)},
    }
    liquid_factor = {"none": 0.0856, "single": 0.0192, "double": None}[shield]
    m = phase == "liquid"
    if liquid_factor is not None:
        corr.loc[m] = P.loc[m] * (1 + liquid_factor * U.loc[m])

    for ph in ("mixed", "solid"):
        a, b, c, umax = pars[shield][ph]
        m = phase == ph
        ce = a * np.exp(-b * U.loc[m].clip(upper=umax)) + c
        corr.loc[m] = P.loc[m] / ce

    out["corrected_P"] = corr
    return out

def save_qc(df, filename):
    df.to_csv(OUT_DIR / filename, index=True)

def missing_percent(df):
    return df.isna().mean().mul(100)

Ausangate

In [11]:
aus_rename = {
    "Air Temp1 HRLY Avg (?)":"Tair", "Air Temp1 HRLY Max(?)":"Tair_max", "Air Temp1 HRLY Min(?)":"Tair_min",
    "Barometric Pressure(mbar)":"PRESS", "Battery Voltage(V)":"Batt_V",
    "In LW Rad. HRLY Avg (W/M-2)":"LWin", "Out LW Rad. HRLY Avg(W/M-2)":"LWout",
    "RH1 HRLY Avg(%)":"RH", "Snow Depth HRLY(cm)":"SnDep",
    "In SW Rad. HRLY Avg (W/m^2)":"SWin", "In SW HRLY Max (W/m^2)":"SWin_max",
    "Out SW Rad. HRLY Avg (W/M-2)":"SWout", "Wind Direction1 HRLY Avg (deg)":"WD",
    "Wind Gust1 (m/s)":"WS_max", "Wind Speed1 HRLY Avg(m/s)":"WS"
}

df_Aus = pd.read_csv(DATA_DIR / "Ausangate_Met_20260722_raw_AK.csv")
df_Aus["TIMESTAMP"] = pd.to_datetime(df_Aus.pop("Date/Time (UTC)"), format="%m/%d/%Y %H:%M", errors="coerce")
df_Aus = hourly_index(df_Aus).rename(columns=aus_rename)
df_Aus[["Tair_max","Tair_min","SWin","SWin_max","SWout"]] = df_Aus[["Tair_max","Tair_min","SWin","SWin_max","SWout"]].apply(pd.to_numeric, errors="coerce")

df_Aus.loc[(df_Aus["Tair_max"] - df_Aus["Tair_min"]) > 5, "Tair_min"] = np.nan
df_Aus.loc[df_Aus["SWin"] < 0, "SWin"] = 0
df_Aus.loc[df_Aus["SWin"] < 0.01, "SWin_max"] = 0
df_Aus.loc[df_Aus["SWin"] < 0.1, "SWout"] = 0
m = (df_Aus["SWout"] / df_Aus["SWin"]) > 0.95
df_Aus.loc[m, "SWin"] = df_Aus.loc[m, "SWout"] / 0.95
df_Aus.loc["2022-09-10 00:00":"2022-10-07 23:00", "WD"] = np.nan
df_Aus.loc["2022-08-08 12:00":"2022-08-10 23:00", "SnDep"] = np.nan
df_Aus["RHi"] = rh_water_to_ice(df_Aus["RH"], df_Aus["Tair"])

aus_cols = ["Tair","Tair_max","Tair_min","RH","RHi","PRESS","WS","WD","WS_max",
            "SWin","SWin_max","SWout","LWin","LWout","SnDep","Batt_V"]
df_Aus = df_Aus[aus_cols]
save_qc(df_Aus, "QC_Ausangate_Met_20260722_AK.csv")

Murmurani Alto

In [12]:
mur_rename = {"AirTC":"Tair", "Pressure":"PRESS", "SoilTemp":"Tsoil", "SoilMois":"SM", "SlrW":"SWin"}

df_Mur = pd.read_csv(DATA_DIR / "Murmurani_Alto_20170709_QCed_AK.csv").iloc[13:].copy()
df_Mur = hourly_index(df_Mur, fmt="%m/%d/%Y %H:%M").rename(columns=mur_rename)
df_Mur.loc["2016-02-18 11:00":"2016-07-11 18:00", "RH"] = np.nan
df_Mur.loc[df_Mur["RH"] < 1, "RH"] = np.nan
df_Mur.loc[df_Mur["SWin"] < 0, "SWin"] = 0
df_Mur.loc[df_Mur.index < "2012-04-04 17:00", "PRESS"] = np.nan
df_Mur["RHi"] = rh_water_to_ice(df_Mur["RH"], df_Mur["Tair"])

df_Mur = df_Mur[["Tair","RH","RHi","PRESS","SWin","Tsoil","SM","SnDep"]]
save_qc(df_Mur, "QC_Murmurani_Alto_20170709_AK.csv")

Osjollo Anante

In [ ]:
df_Osj = pd.read_csv(DATA_DIR / "Osjollo_Anante_hr_20150721_QCed_AK.csv")
df_Osj = hourly_index(df_Osj).rename(columns={"AirTC":"Tair"})
df_Osj.loc[df_Osj["WS"] == 0, ["WS","WD"]] = np.nan
df_Osj["RHi"] = rh_water_to_ice(df_Osj["RH"], df_Osj["Tair"])

df_Osj = df_Osj[["Tair","RH","RHi","WS","WD"]]
save_qc(df_Osj, "1QC_Osjollo_Anante_hr_20150721_AK.csv")

Quelccaya

In [ ]:
q_rename = {
    "AirTC":"Tair", "synop":"Synop", "Precip_Tot":"Precip", "Tot24":"Precip24",
    "WS_Max":"WS_max", "SlrW":"SWinX", "SlrW_Max":"SWin_max", "SlrW_Avg":"SWin",
    "Bucket_Tot":"Bucket"
}

df_Que = pd.read_csv(DATA_DIR / "Quelccaya_hr_20250928_QCed_AK.csv")
df_Que = hourly_index(df_Que).rename(columns=q_rename)
df_Que[["SWin","SWin_max","Precip","Precip24"]] = df_Que[["SWin","SWin_max","Precip","Precip24"]].apply(pd.to_numeric, errors="coerce")
df_Que.loc[df_Que["WS"] == 0, "WD"] = np.nan
df_Que["RHi"] = rh_water_to_ice(df_Que["RH"], df_Que["Tair"])

bad_dates = pd.to_datetime([
    "2023-10-26","2023-11-28","2023-12-02","2024-01-15","2024-01-27","2024-01-30",
    "2024-01-31","2024-02-01","2024-02-03","2024-02-04","2024-02-13","2024-02-14",
    "2024-02-25","2024-03-23","2024-05-09"
])
df_Que.loc[df_Que.index.normalize().isin(bad_dates), "Precip"] = np.nan
df_Que = correct_precipitation(df_Que, shield="double")

que_cols = ["Tair","RH","RHi","WS","WD","WS_max","SWin","SWin_max","SnDep",
            "Precip","Precip24","corrected_P","modelled_phase","Bucket"]
df_Que = df_Que[que_cols]
save_qc(df_Que, "QC_Quelccaya_hr_20250928_AK.csv")

Lake Sibinacocha

In [ ]:
sib_rename = {
    "AirTC":"Tair", "synop":"Synop", "Precip_Tot":"Precip", "Tot24":"Precip24",
    "WS_Max":"WS_max", "SlrW":"SWinX", "SlrW_Max":"SWin_max", "SlrW_Avg":"SWin",
    "Bucket_Tot":"Bucket"
}

df_Sib = pd.read_csv(DATA_DIR / "Sibinacocha_20260816_QCed_AK.csv")
df_Sib = hourly_index(df_Sib).rename(columns=sib_rename)
df_Sib[["Precip","Precip24","SWin","SWin_max"]] = df_Sib[["Precip","Precip24","SWin","SWin_max"]].apply(pd.to_numeric, errors="coerce")
df_Sib.loc[df_Sib["SnDep"] < -0.25, "SnDep"] = np.nan
df_Sib.loc[df_Sib["RH"] < 1, "RH"] = np.nan
df_Sib["RHi"] = rh_water_to_ice(df_Sib["RH"], df_Sib["Tair"])
df_Sib = df_Sib.drop(columns=["WS","WD","WS_max","SWinX"], errors="ignore")

# Separate wind record used for precipitation correction.
df_Sib_ws = pd.read_csv(DATA_DIR / "sibinacocha_WS.csv").rename(columns={"Timestamp":"TIMESTAMP","WindSpeed":"WS","WindDir":"WD"})
df_Sib_ws = hourly_index(df_Sib_ws)[["WS","WD"]]
df_Sib_ws["WS"] *= 0.5  # Site-specific factor used in the original workflow.
df_Sib = df_Sib.join(df_Sib_ws, how="left")

# Manuscript describes the Sibinacocha Pluvio² as single-Alter shielded.
df_Sib = correct_precipitation(df_Sib, shield="single")

sib_cols = ["Tair","RH","RHi","Precip","Precip24","corrected_P","modelled_phase",
            "SWin","SWin_max","SnDep","Bucket","WS","WD"]
df_Sib = df_Sib[sib_cols]
save_qc(df_Sib, "QC_Sibinacocha_20260816_AK.csv")

Data completeness and hydrological-year precipitation

In [ ]:
stations = {"Ausangate":df_Aus, "Murmurani Alto":df_Mur, "Osjollo Anante":df_Osj,
            "Quelccaya":df_Que, "Sibinacocha":df_Sib}

completeness = pd.concat({name: 100 - missing_percent(df) for name, df in stations.items()}, axis=1).round(1)
completeness.to_csv(OUT_DIR / "1AWS_data_completeness_percent.csv")

df_Sib["HydroYear"] = df_Sib.index.year + (df_Sib.index.month >= 9)
hydro_precip = df_Sib.groupby("HydroYear")[["Precip","corrected_P"]].sum(min_count=1).round(1)

completeness, hydro_precip

(                Ausangate  Murmurani Alto  Osjollo Anante  Quelccaya  \
 Tair                 99.1           100.0           100.0       99.5   
 Tair_max             99.1             NaN             NaN        NaN   
 Tair_min             95.6             NaN             NaN        NaN   
 RH                   99.1            92.4           100.0       95.7   
 RHi                  99.1            92.4           100.0       95.7   
 PRESS                99.1           100.0             NaN        NaN   
 WS                   99.1             NaN            82.8       99.5   
 WD                   75.7             NaN            82.8       98.4   
 WS_max               99.1             NaN             NaN       99.5   
 SWin                 99.1            99.7             NaN       97.0   
 SWin_max             99.1             NaN             NaN       97.0   
 SWout                99.1             NaN             NaN        NaN   
 LWin                 99.1             NaN         